# 04 — Comparison: elastic-net vs. Bayesian PTGS

Head-to-head on **identical** nested-CV folds (the shared harness guarantees the same splits,
so the comparison is paired per fold). Uses `run_benchmark` + the `viz` figures. All logic is
in `ptgs_bc`; this notebook orchestrates and shows results.

## Setup

In [ ]:
import ptgs_bc as ptgs
from ptgs_bc import (simulate_dataset, ElasticNetBuilder, BayesBuilder,
                     run_benchmark, summary_table, per_fold_table, plot_performance)
%matplotlib inline

## Data + the two arms

In [ ]:
ds, true_w = simulate_dataset(n_samples=400, n_genes=100, n_causal=15, family="gaussian", seed=0)
builders = [
    ElasticNetBuilder(),
    BayesBuilder(prior="regularized_horseshoe", num_warmup=300, num_samples=300),
]
print(ds.n_samples, "samples x", ds.n_genes, "genes | family:", ds.family)

## Run both through the same nested CV

In [ ]:
res = run_benchmark(builders, ds, outer_k=5, seed=0)
summary_table(res)          # mean +/- sd per arm

## Per-fold (paired) table

In [ ]:
pf = per_fold_table(res)
pf.pivot(index="fold", columns="builder", values="value")

## Performance comparison figure

Left: mean ± sd per arm with the individual folds. Right: per-fold arm-vs-arm scatter with a
y=x line (points above the line = the y-axis arm wins that fold).

In [ ]:
fig = plot_performance(res)
fig

## Paired difference (quick summary)

Because the folds are identical, differences are paired. A formal paired test (e.g.
`scipy.stats.wilcoxon`, as in the reference) can be added here if SciPy is available.

In [ ]:
import numpy as np
names = list(res)
a, b = res[names[0]].values, res[names[1]].values
print(f"{names[1]} - {names[0]} per-fold diff: mean={np.mean(b-a):+.4f}, "
      f"wins={int((b>a).sum())}/{len(a)}")